# Go / No-Go: does the regime $\alpha_H \ll \kappa$ occur in nature?

**Follow-up to** *When Do Predictions Help Streaming Subgraph Counting?* (Array 32 (2026) 101199).
**Target:** Data Mining and Knowledge Discovery (Springer).

That paper showed, on eleven SNAP graphs, that the oracle-width stays a constant fraction of the
degeneracy, $\alpha_H/\kappa \in [0.52, 0.89]$, so predictions buy a large constant and never an
exponent. The separation of Theorem 11 needs $\alpha_H/\kappa \to 0$: a **dense region that bears no
copies**. This notebook hunts for that region in three places the earlier study never looked.

| Track | Object | Pattern | Why it might work |
|---|---|---|---|
| A | incidence graph of a hypergraph | $C_4$ (butterfly) | a partial linear space is $C_4$-free but dense |
| **A2** | **projected graph of a hypergraph** | **cross-hyperedge triangle** | **a big hyperedge drops a clique bearing no cross-copies** |
| B | temporal window of a stream | $K_3$ | hubs are active before triangles close |

Track A2 is the one to watch; section 2 explains why A alone cannot deliver.

**Decision rule.** A family supports a paper only if, jointly:

1. $\alpha_H/\kappa$ well below the published band $[0.52, 0.89]$ — target $< 0.2$;
2. the ratio **decreases** with $m$ (or with window length), not merely small at one size;
3. non-degenerate counting problem: copy density $\#H/m$ bounded away from $0$.

Everything is exact: $\kappa$ by peeling, $\alpha_H$ by max-flow. No sampling, no heuristics.

## 0. Setup and measurement library

$\alpha_H$ is the pseudoarboricity of the copy-bearing subgraph: the least $k$ for which those edges
admit an orientation of maximum out-degree $k$. Computed **exactly** by binary search over $k$ with a
max-flow feasibility test (Hakimi / Frank-Gyarfas): $s \to$ edge (cap 1), edge $\to$ each endpoint
(cap 1), vertex $\to t$ (cap $k$); feasible iff the flow saturates all $m$ edges.

In [ ]:
"""Core measurement library: kappa, alpha_H, copy-bearing subgraphs, diagnostics."""
import gzip
import io
import os
import urllib.request
from collections import defaultdict

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_flow

# ----------------------------------------------------------------- graph basics


def relabel(edges):
    """Map arbitrary hashable node labels to consecutive integers."""
    ids, out = {}, []
    for u, v in edges:
        for x in (u, v):
            if x not in ids:
                ids[x] = len(ids)
        out.append((ids[u], ids[v]))
    return out, ids


def canon(edges):
    """De-duplicate, drop self-loops, relabel to ints, return sorted (u,v), u<v."""
    edges, _ = relabel(edges)
    s = set()
    for u, v in edges:
        if u == v:
            continue
        s.add((u, v) if u < v else (v, u))
    return sorted(s)


def build_adj(edges):
    adj = defaultdict(set)
    for u, v in edges:
        if u == v:
            continue
        adj[u].add(v)
        adj[v].add(u)
    return adj


def degeneracy(adj):
    """Exact degeneracy (k-core number) by peeling.  O(n + m)."""
    if not adj:
        return 0, {}
    deg = {v: len(adj[v]) for v in adj}
    maxdeg = max(deg.values())
    buckets = [set() for _ in range(maxdeg + 1)]
    for v, d in deg.items():
        buckets[d].add(v)
    core, k, removed = {}, 0, set()
    for _ in range(len(deg)):
        i = 0
        while i <= maxdeg and not buckets[i]:
            i += 1
        if i > maxdeg:
            break
        v = buckets[i].pop()
        k = max(k, i)
        core[v] = k
        removed.add(v)
        for w in adj[v]:
            if w in removed:
                continue
            d = deg[w]
            buckets[d].discard(w)
            deg[w] = d - 1
            buckets[d - 1].add(w)
    return k, core


# -------------------------------------------------- oracle-width (pseudoarboricity)


def _orientation_feasible(edges, nodes, k):
    """Does an orientation with max out-degree <= k exist?  Max-flow feasibility test.

    s -> edge (cap 1); edge -> each endpoint (cap 1); vertex -> t (cap k).
    Feasible iff maxflow == |E|  (Hakimi / Frank-Gyarfas).
    """
    m, n = len(edges), len(nodes)
    if m == 0:
        return True
    idx = {v: i for i, v in enumerate(nodes)}
    S, E0, V0 = 0, 1, 1 + m
    T = 1 + m + n
    rows, cols, data = [], [], []
    for i, (u, v) in enumerate(edges):
        rows.append(S); cols.append(E0 + i); data.append(1)
        rows.append(E0 + i); cols.append(V0 + idx[u]); data.append(1)
        rows.append(E0 + i); cols.append(V0 + idx[v]); data.append(1)
    for i in range(n):
        rows.append(V0 + i); cols.append(T); data.append(int(k))
    g = csr_matrix((np.array(data, dtype=np.int32),
                    (np.array(rows), np.array(cols))), shape=(T + 1, T + 1))
    return int(maximum_flow(g, S, T).flow_value) == m


def pseudoarboricity(edges):
    """Exact min-max-out-degree orientation number of an edge set."""
    edges = canon(edges)
    if not edges:
        return 0
    nodes = sorted({x for e in edges for x in e})
    hi = max(degeneracy(build_adj(edges))[0], 1)
    lo = 1
    while lo < hi:
        mid = (lo + hi) // 2
        if _orientation_feasible(edges, nodes, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


# --------------------------------------------------------------- copy-bearing sets


def copy_bearing_K3(adj):
    """Edges on >= 1 triangle, plus #K3."""
    out, tri = [], 0
    for u in adj:
        for v in adj[u]:
            if u >= v:
                continue
            c = len(adj[u] & adj[v])
            if c:
                out.append((u, v))
                tri += c
    return out, tri // 3


def codegree_map(adj, deg_cap=None):
    """codeg[(a,b)] = |N(a) & N(b)| for pairs with a common neighbour."""
    codeg = defaultdict(int)
    for w in adj:
        nb = sorted(adj[w])
        if deg_cap is not None and len(nb) > deg_cap:
            continue
        for i in range(len(nb)):
            a = nb[i]
            for j in range(i + 1, len(nb)):
                codeg[(a, nb[j])] += 1
    return codeg


def copy_bearing_C4(adj, deg_cap=None):
    """Edges on >= 1 four-cycle (exact), plus #C4.

    Edge (u,v) lies on a C4 iff some u' in N(v)\\{u} has |N(u) & N(u')| >= 2.
    """
    codeg = codegree_map(adj, deg_cap)
    heavy = {p for p, c in codeg.items() if c >= 2}
    n_c4 = sum(c * (c - 1) // 2 for c in codeg.values()) // 2
    out = []
    for u in adj:
        for v in adj[u]:
            if u >= v:
                continue
            for up in adj[v]:
                if up == u:
                    continue
                p = (u, up) if u < up else (up, u)
                if p in heavy:
                    out.append((u, v))
                    break
    return out, n_c4


# ------------------------------------------------------------------- diagnostic


def diagnostic(edges, pattern="K3", label="", deg_cap=None, exact_alpha=True):
    """Return the full row: n, m, kappa, kappa_copy, alpha_H, ratios, copy density."""
    edges = canon(edges)
    adj = build_adj(edges)
    kappa, _ = degeneracy(adj)
    if pattern == "K3":
        cb, n_copies = copy_bearing_K3(adj)
    elif pattern == "C4":
        cb, n_copies = copy_bearing_C4(adj, deg_cap)
    else:
        raise ValueError(pattern)
    adj_cb = build_adj(cb)
    kappa_copy, _ = degeneracy(adj_cb)
    if exact_alpha:
        alpha = pseudoarboricity(cb)
    else:  # cheap two-sided bracket, no max-flow
        alpha = None
    row = dict(
        label=label, pattern=pattern, n=len(adj), m=len(edges),
        kappa=kappa, kappa_copy=kappa_copy, alpha=alpha,
        m_copy=len(cb), n_copies=n_copies,
        copy_density=(n_copies / len(edges)) if edges else 0.0,
        alpha_over_kappa=(alpha / kappa) if (alpha is not None and kappa) else None,
        kappa_copy_over_kappa=(kappa_copy / kappa) if kappa else None,
        # structural bracket:  ceil(kappa_copy/2) <= alpha_H <= kappa_copy <= kappa
        alpha_lb=int(np.ceil(kappa_copy / 2)),
        alpha_ub=kappa_copy,
    )
    return row


# ------------------------------------------------------------------ constructions


def affine_plane_incidence(q):
    """Incidence graph of AG(2,q), q prime.  C4-free, degeneracy Theta(q)=Theta(sqrt(m))."""
    E = []
    for a in range(q):
        for b in range(q):
            lid = ("L", a, b)
            for x in range(q):
                E.append((("P", x, (a * x + b) % q), lid))
    for c in range(q):
        lid = ("V", c)
        for y in range(q):
            E.append((("P", c, y), lid))
    return E


def steiner_plus_butterflies(q, n_gadgets, r=3):
    """AG(2,q) copy-free dense core  +  n_gadgets shallow butterfly gadgets.

    Natural analogue of the friendship+K_{d,d} instance of Theorem 11, with the
    copy-free dense region replaced by a genuine partial linear space.
    """
    E = affine_plane_incidence(q)
    for g in range(n_gadgets):
        a, b = ("G", g, 0), ("G", g, 1)
        for t in range(2):
            h = ("H", g, t)
            E.append((a, h))
            E.append((b, h))
            for j in range(r - 2):
                E.append((("F", g, t, j), h))
    return E


def friendship_plus_decoy(s):
    """The instance of Theorem 11: F_s  U  K_{d,d}, d = ceil(sqrt(s))."""
    d = int(np.ceil(np.sqrt(s)))
    E = []
    for i in range(s):
        E += [("h", ("a", i)), ("h", ("b", i)), (("a", i), ("b", i))]
    for i in range(d):
        for j in range(d):
            E.append((("L", i), ("R", j)))
    return E


def random_uniform_hypergraph(n, m_h, r, seed=0):
    rng = np.random.default_rng(seed)
    return [tuple(rng.choice(n, size=r, replace=False)) for _ in range(m_h)]


def incidence_edges(hyperedges):
    """Bipartite incidence graph of a hyperedge list."""
    E = []
    for i, h in enumerate(hyperedges):
        for v in set(h):
            E.append((("v", v), ("e", i)))
    return E


# ----------------------------------------------------------------------- loaders


def fetch(url, dest):
    if os.path.exists(dest):
        return dest
    urllib.request.urlretrieve(url, dest)
    return dest


def load_temporal(path):
    """SNAP temporal edge list: 'u v t' per line.  Returns sorted (t,u,v) array."""
    op = gzip.open if path.endswith(".gz") else open
    rows = []
    with op(path, "rt") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            p = line.split()
            if len(p) < 3:
                continue
            rows.append((int(p[2]), int(p[0]), int(p[1])))
    rows.sort()
    return rows


def load_hyperedges_simplices(nverts_path, simplices_path):
    """Benson ARB format: one size per line + a flat vertex stream."""
    op1 = gzip.open if nverts_path.endswith(".gz") else open
    op2 = gzip.open if simplices_path.endswith(".gz") else open
    sizes = [int(x) for x in op1(nverts_path, "rt").read().split()]
    flat = [int(x) for x in op2(simplices_path, "rt").read().split()]
    out, i = [], 0
    for s in sizes:
        out.append(tuple(flat[i:i + s]))
        i += s
    return out


def load_hyperedges_lines(path):
    """One hyperedge per line, whitespace- or comma-separated vertex ids."""
    op = gzip.open if path.endswith(".gz") else open
    out = []
    with op(path, "rt") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            out.append(tuple(line.replace(",", " ").split()))
    return out


# ------------------------------------- higher-order pattern: cross-hyperedge triangles

def projection(hyperedges, max_size=25):
    """Projected graph of a hypergraph + the membership map v -> set of hyperedge ids."""
    memb = defaultdict(set)
    E = set()
    for i, h in enumerate(hyperedges):
        s = sorted(set(h))
        if len(s) < 2 or len(s) > max_size:
            continue
        for v in s:
            memb[v].add(i)
        for a in range(len(s)):
            for b in range(a + 1, len(s)):
                E.add((s[a], s[b]))
    return sorted(E), memb


def copy_bearing_cross_K3(adj, memb):
    """Edges on >= 1 triangle NOT contained in a single hyperedge, plus the count.

    A triangle {u,v,w} is *cross* iff memb[u] & memb[v] & memb[w] is empty: the three
    vertices never co-occur in one hyperedge, so the triangle is genuine higher-order
    triadic closure rather than the clique a single hyperedge drops on the projection.
    """
    out, n_cross = [], 0
    for u in adj:
        for v in adj[u]:
            if u >= v:
                continue
            common = adj[u] & adj[v]
            if not common:
                continue
            muv = memb[u] & memb[v]
            hit = False
            for w in common:
                if not (muv & memb[w]):
                    n_cross += 1
                    hit = True
            if hit:
                out.append((u, v))
    return out, n_cross // 3


def diagnostic_hypergraph(hyperedges, label='', max_size=25, exact_alpha=True):
    """Diagnostic for cross-hyperedge triangles on the projected graph."""
    E, memb = projection(hyperedges, max_size)
    adj = build_adj(E)
    kappa, _ = degeneracy(adj)
    cb, n_cross = copy_bearing_cross_K3(adj, memb)
    kappa_copy, _ = degeneracy(build_adj(cb))
    alpha = pseudoarboricity(cb) if exact_alpha else None
    return dict(
        label=label, pattern='cross-K3', n=len(adj), m=len(E),
        kappa=kappa, kappa_copy=kappa_copy, alpha=alpha,
        m_copy=len(cb), n_copies=n_cross,
        copy_density=n_cross / len(E) if E else 0.0,
        alpha_over_kappa=(alpha / kappa) if (alpha is not None and kappa) else None,
        kappa_copy_over_kappa=(kappa_copy / kappa) if kappa else None,
        alpha_lb=int(np.ceil(kappa_copy / 2)), alpha_ub=kappa_copy,
        max_hyperedge=max((len(set(h)) for h in hyperedges), default=0),
    )


def cliques_plus_cross(n_cliques, clique_size, n_gadgets, seed=0):
    """Synthetic positive control: big copy-free cliques + shallow cross-triangle gadgets."""
    rng = np.random.default_rng(seed)
    H, nxt = [], 0
    cliques = []
    for _ in range(n_cliques):
        c = list(range(nxt, nxt + clique_size)); nxt += clique_size
        H.append(tuple(c)); cliques.append(c)
    for g in range(n_gadgets):
        c = cliques[rng.integers(len(cliques))]
        u, v = rng.choice(c, size=2, replace=False)
        w = nxt; nxt += 1
        H.append((int(u), w))
        H.append((int(v), w))
    return H


## 1. The structural bracket — why $0.52$ was never going to be beaten

Write $G_{\text{copy}}$ for the subgraph of copy-bearing edges and $\kappa_{\text{copy}}$ for its
degeneracy. Two standard facts pin $\alpha_H$ between them:

$$\left\lceil \kappa_{\text{copy}}/2 \right\rceil \;\le\; \alpha_H \;\le\; \kappa_{\text{copy}} \;\le\; \kappa .$$

*Upper:* a degeneracy ordering orients every edge towards the later endpoint, out-degree
$\le \kappa_{\text{copy}}$. *Lower:* a graph of degeneracy $k$ contains a subgraph of minimum degree
$k$, hence of density $\ge k/2$, and pseudoarboricity is at least the maximum density.

**Consequence.** If the copy-bearing edges retain the dense core, i.e. $\kappa_{\text{copy}} = \kappa$,
then $\alpha_H/\kappa \ge \lceil \kappa/2 \rceil / \kappa \approx 0.5$ **automatically**. The published
floor of $0.52$ is close to an arithmetic floor, not an empirical accident — and the whole search
collapses to one cheap question,

$$\kappa_{\text{copy}}/\kappa \to 0 \;?$$

answerable in $O(m)$ by two peelings, no max-flow. That is the screening statistic used throughout,
and it is a lemma worth stating in the new paper in its own right: it converts the Array paper's
empirical regularity into a structural explanation.

In [ ]:
import pandas as pd, numpy as np, networkx as nx, matplotlib.pyplot as plt
pd.set_option('display.width', 220)

controls = []
for s in [400, 1600, 6400]:
    controls.append(diagnostic(friendship_plus_decoy(s), 'K3', label=f'Thm-11 gadget (s={s})'))
for n, k in [(2000, 4), (4000, 6)]:
    controls.append(diagnostic(list(nx.barabasi_albert_graph(n, k, seed=1).edges()), 'K3',
                               label=f'BA({n},{k})'))
controls.append(diagnostic(list(nx.powerlaw_cluster_graph(3000, 4, 0.4, seed=1).edges()), 'K3',
                           label='powerlaw-cluster'))

cdf = pd.DataFrame(controls)
assert all(r.alpha_lb <= r.alpha <= r.alpha_ub <= r.kappa for r in cdf.itertuples()), 'bracket violated'
print('structural bracket holds on all controls\n')
cdf[['label','m','kappa','kappa_copy','alpha_lb','alpha','alpha_ub',
     'alpha_over_kappa','kappa_copy_over_kappa','copy_density']].round(3)

Read this table before going further. The Theorem-11 gadget is the **positive control**:
$\alpha = 2$ exactly, ratio $\to 0$, copy density $0.25$. The BA and powerlaw-cluster rows are the
**negative control**: the copy-bearing edges keep the whole core, $\kappa_{\text{copy}} = \kappa$,
ratio $=1$. Any real dataset behaving like the first row is a paper.

## 2. Track A: incidence structures realize the separation — and hit a ceiling

$AG(2,q)$, the affine plane over $\mathbb{F}_q$: $q^2$ points, $q^2+q$ lines of $q$ points each, any
two points on exactly one line. Its incidence graph is $C_4$-free with degeneracy $\Theta(q) =
\Theta(\sqrt{m})$ — a dense region bearing **no** copies, arising from geometry rather than from a
$K_{d,d}$ glued on by hand. $C_4$-freeness alone is vacuous for counting ($\#C_4=0$), so we add
shallow butterfly gadgets: pairs co-occurring in two hyperedges, disjoint from one another.

In [ ]:
rowsA = []
for q in [7, 11, 13, 17]:
    rowsA.append(diagnostic(affine_plane_incidence(q), 'C4', label=f'AG(2,{q}) pure'))
for q in [7, 11, 13, 17, 19]:
    g = 8 * q * q // 3
    rowsA.append(diagnostic(steiner_plus_butterflies(q, g), 'C4',
                            label=f'AG(2,{q}) + {g} butterflies'))
A = pd.DataFrame(rowsA)
sub = A[A.label.str.contains('butterflies')]
b = np.polyfit(np.log(sub.m), np.log(sub.alpha_over_kappa), 1)[0]

plt.figure(figsize=(5.2,3.6))
plt.loglog(sub.m, sub.alpha_over_kappa, 'o-', label=f'AG(2,q)+gadgets, slope={b:.2f}')
plt.axhspan(0.52, 0.89, color='crimson', alpha=.15, label='Array band [0.52, 0.89]')
plt.xlabel('m'); plt.ylabel(r'$\alpha_{C_4}/\kappa$'); plt.legend(fontsize=8)
plt.title('Separation regime from a partial linear space'); plt.tight_layout(); plt.show()
print(f'ratio decays as m^{b:.3f}')
A[['label','m','kappa','kappa_copy','alpha','alpha_over_kappa',
   'kappa_copy_over_kappa','n_copies','copy_density']].round(3)

### Why this track cannot carry the paper on its own

In an incidence graph every hyperedge-node has degree $|e|$, so peeling the hyperedge side first
gives

$$\kappa(\text{incidence graph}) \;\le\; \max_{e} |e| .$$

Real hypergraph collections cap simplex size at 25, so $\kappa = O(1)$ there and $\alpha_H/\kappa$
**cannot** decay polynomially in $m$ however Steiner-like the data is. The $\Theta(\sqrt m)$
degeneracy of $AG(2,q)$ came precisely from lines whose size grows with $q$.

A second lemma, and a useful negative one: it rules out an entire class of candidate domains in one
line. It also points at the fix — stop looking at the incidence graph.

## 3. Track A2: cross-hyperedge triangles on the projected graph

Project the hypergraph: $u \sim v$ iff some hyperedge contains both. A hyperedge of size $r$ drops a
clique $K_r$, so $\kappa \ge r-1$ — the projection has genuinely dense regions. Now count only
**cross-hyperedge triangles**: triples $\{u,v,w\}$ that are pairwise connected but never co-occur in
a single hyperedge. These are exactly the higher-order triadic closures the simplicial-closure
literature cares about, and a natural data-mining primitive in their own right.

The point: *the clique a large hyperedge drops bears no cross-triangles at all.* It is dense and
copy-free — the decoy of Theorem 11, occurring naturally, with nothing constructed. If cross-triangles
sit on a shallow substructure while the big hyperedges supply the degeneracy, then
$\alpha_H \ll \kappa$ and predictions change the exponent.

Synthetic positive control first: disjoint hyperedges of size $r$ (copy-free cliques) plus shallow
cross gadgets.

In [ ]:
rowsX = []
for cs in [20, 40, 60, 80]:
    H = cliques_plus_cross(6, cs, 6 * cs, seed=1)
    rowsX.append(diagnostic_hypergraph(H, label=f'cliques(r={cs}) + cross gadgets', max_size=200))
X = pd.DataFrame(rowsX)
bx = np.polyfit(np.log(X.m), np.log(X.alpha_over_kappa), 1)[0]
print(f'ratio decays as m^{bx:.3f}')
X[['label','m','kappa','kappa_copy','alpha','alpha_over_kappa',
   'kappa_copy_over_kappa','n_copies','copy_density']].round(3)

## 4. Real hypergraphs

Benson's collection (`cs.cornell.edu/~arb/data`) hosts the files on Google Drive, which is why plain
`urlretrieve` failed — use `gdown` with the file ids below. Each archive is a `.tar.gz` containing
`<name>-nverts.txt` and `<name>-simplices.txt`.

`congress-bills` and `DAWN` are the interesting ones for Track A2: large hyperedges, hence a large
$\kappa$ in the projection. `contact-primary-school` and `email-Enron` are small negative controls.

In [ ]:
!pip -q install gdown

import gdown, tarfile, glob, os

ARB_IDS = {
    'NDC-classes':            '1tpDiP1c73O18gCYEx4OI7kx8V_IdYxLt',
    'contact-primary-school': '1sBHSEIyvVKavAho524Ro4cKL66W6rn-t',
    'email-Enron':            '1tTVZkdpgRW47WWmsrdUCukHz0x2M6N77',
    'tags-math-sx':           '1eDevpF6EZs19rLouNpiKGLIlFOLUfKKG',
    'congress-bills':         '1gH1uJMZpn_SCJSRbORPH4JRQeevLwTyO',
    'DAWN':                   '1wGwoG7oBWnNN7J9TEpjqNpODbsYfMxp4',
    # unrestricted versions: bigger hyperedges -> bigger kappa in the projection
    'congress-bills-full':    '1FYu5367ijQLDjyGigbf709YpbjocYxif',
    'email-Enron-full':       '118tda4_E7ebtTNhKj12IJ9Y71rtJ7UP6',
}

# Download + extract an ARB hypergraph; returns the hyperedge list or None.
def fetch_arb(name):
    tgz = f'{name}.tar.gz'
    try:
        if not os.path.exists(tgz):
            gdown.download(id=ARB_IDS[name], output=tgz, quiet=True)
        with tarfile.open(tgz) as t:
            t.extractall('.')
        nv = glob.glob(f'**/{name}-nverts.txt', recursive=True)
        sp = glob.glob(f'**/{name}-simplices.txt', recursive=True)
        if not (nv and sp):
            nv = glob.glob('**/*-nverts.txt', recursive=True)
            sp = glob.glob('**/*-simplices.txt', recursive=True)
        return load_hyperedges_simplices(nv[0], sp[0])
    except Exception as e:
        print(f'  {name}: {type(e).__name__} - {e}')
        return None

WANT = ['contact-primary-school', 'email-Enron', 'NDC-classes', 'DAWN', 'congress-bills']
HYPER = {}
for name in WANT:
    h = fetch_arb(name)
    if h:
        HYPER[name] = h
        sizes = [len(set(x)) for x in h]
        print(f'{name}: {len(h)} simplices, max size {max(sizes)}, mean {np.mean(sizes):.1f}')

if not HYPER:
    print('Nothing downloaded. Upload a hyperedge-per-line file and run:')
    print("    HYPER['mydata'] = load_hyperedges_lines('mydata.txt')")

In [ ]:
# Track A2 on real data.  De-duplicate simplices first: repeated identical hyperedges add
# no structure but blow up the membership sets.
MAX_SIMPLICES = 40_000        # prefix cap per dataset; raise if Colab is comfortable
MAX_SIZE      = 25

rowsR2 = []
for name, h in HYPER.items():
    uniq = list(dict.fromkeys(tuple(sorted(set(x))) for x in h))
    uniq = [x for x in uniq if 2 <= len(x) <= MAX_SIZE][:MAX_SIMPLICES]
    print(f'{name}: {len(uniq)} unique simplices ... ', end='', flush=True)
    r = diagnostic_hypergraph(uniq, label=name, max_size=MAX_SIZE)
    rowsR2.append(r)
    print(f"kappa={r['kappa']}  alpha={r['alpha']}  ratio={r['alpha_over_kappa']:.3f}  "
          f"copy_density={r['copy_density']:.3f}")

R2 = pd.DataFrame(rowsR2)
R2[['label','m','kappa','kappa_copy','alpha','alpha_over_kappa',
    'kappa_copy_over_kappa','n_copies','copy_density','max_hyperedge']].round(3) if rowsR2 else 'none'

In [ ]:
# Track A (butterflies on the incidence graph) on the same data, for the comparison table.
# Expect kappa <= max hyperedge size, i.e. the ceiling of section 2.
rowsR = []
for name, h in HYPER.items():
    uniq = list(dict.fromkeys(tuple(sorted(set(x))) for x in h))
    uniq = [x for x in uniq if 2 <= len(x) <= MAX_SIZE][:MAX_SIMPLICES]
    r = diagnostic(incidence_edges(uniq), 'C4', label=name + ' (incidence)', deg_cap=400)
    rowsR.append(r)
    print(f"{name}: kappa={r['kappa']} alpha={r['alpha']} ratio={r['alpha_over_kappa']:.3f}")
R = pd.DataFrame(rowsR)
R[['label','m','kappa','kappa_copy','alpha','alpha_over_kappa','copy_density']].round(3) if rowsR else 'none'

**How to read these two tables.** The deciding column is `kappa_copy_over_kappa`. If it is $1.0$,
the cross-triangles live on the same dense core as everything else and this domain is another
instance of the Array paper's negative finding. If it drops towards $0$ while `copy_density` stays
healthy, you have the first natural family in which predictions can change the exponent — and the
incidence table beside it shows why the obvious first attempt (Track A) could not have found it.

## 5. Track B: temporal windows

Slice a timestamped stream into windows of increasing length and measure the diagnostic per window.
Hypothesis: in a short window the degree hubs are present but the triangles have not closed yet, so
$\kappa_{\text{copy}} \ll \kappa$; as the window grows the graph converges to the static snapshot and
the ratio climbs back into the published band.

In [ ]:
TEMPORAL = {
    'CollegeMsg':             'https://snap.stanford.edu/data/CollegeMsg.txt.gz',
    'email-Eu-core-temporal': 'https://snap.stanford.edu/data/email-Eu-core-temporal.txt.gz',
    'sx-mathoverflow':        'https://snap.stanford.edu/data/sx-mathoverflow.txt.gz',
}
streams = {}
for name, url in TEMPORAL.items():
    try:
        streams[name] = load_temporal(fetch(url, name + '.txt.gz'))
        print(f'{name}: {len(streams[name])} timestamped edges')
    except Exception as e:
        print(f'{name}: {type(e).__name__} - {e}')

In [ ]:
DAY = 86400
WINDOWS = [1*DAY, 3*DAY, 7*DAY, 30*DAY, 90*DAY, 365*DAY, 10**9]   # last = whole stream
N_WIN = 6

def window_sweep(rows, windows=WINDOWS, n_win=N_WIN, min_m=300, max_m=200_000):
    ts = np.array([r[0] for r in rows])
    t0, t1 = ts[0], ts[-1]
    out = []
    for W in windows:
        span = min(W, t1 - t0)
        for s in sorted(set(np.linspace(t0, max(t0, t1 - span), n_win).astype(int))):
            i, j = np.searchsorted(ts, s), np.searchsorted(ts, s + span)
            E = canon([(u, v) for _, u, v in rows[i:j]])
            if not (min_m <= len(E) <= max_m):
                continue
            r = diagnostic(E, 'K3', label=f'W={span/DAY:.1f}d')
            r['window_days'] = span / DAY
            out.append(r)
    return pd.DataFrame(out)

T = {}
for name, rows in streams.items():
    print(f'--- {name}')
    T[name] = window_sweep(rows)
    if len(T[name]):
        print(T[name].groupby('window_days')[['m','kappa','kappa_copy','alpha_over_kappa',
              'kappa_copy_over_kappa','copy_density']].mean().round(3))

In [ ]:
plt.figure(figsize=(6,4))
for name, df in T.items():
    if not len(df):
        continue
    g = df.groupby('window_days')['alpha_over_kappa'].agg(['mean','std'])
    plt.errorbar(g.index, g['mean'], yerr=g['std'].fillna(0), marker='o', capsize=3, label=name)
plt.axhspan(0.52, 0.89, color='crimson', alpha=.15, label='Array band')
plt.xscale('log'); plt.xlabel('window length (days)'); plt.ylabel(r'$\alpha_{K_3}/\kappa$')
plt.ylim(0, 1.05); plt.legend(fontsize=8)
plt.title('Does slicing the stream open the gap?'); plt.tight_layout(); plt.show()

## 6. Baseline check: reproduce the Array band, and add the new column

Optional, worth running once. Two things to confirm on static SNAP graphs: that `alpha_over_kappa`
lands in $[0.52, 0.89]$ (validates the implementation against the published table) and that
`kappa_copy_over_kappa` is $1.0$ (validates the lemma of section 1 as the explanation for the band).

In [ ]:
STATIC = {
    'ca-GrQc':  'https://snap.stanford.edu/data/ca-GrQc.txt.gz',
    'ca-HepTh': 'https://snap.stanford.edu/data/ca-HepTh.txt.gz',
    'facebook': 'https://snap.stanford.edu/data/facebook_combined.txt.gz',
}

def load_edge_list(path):
    op = gzip.open if path.endswith('.gz') else open
    E = []
    with op(path, 'rt') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            a, b = line.split()[:2]
            E.append((a, b))
    return E

rowsS = []
for name, url in STATIC.items():
    try:
        r = diagnostic(load_edge_list(fetch(url, name + '.txt.gz')), 'K3', label=name)
        rowsS.append(r)
        print(f"{name}: alpha/kappa={r['alpha_over_kappa']:.3f}  "
              f"kappa_copy/kappa={r['kappa_copy_over_kappa']:.3f}")
    except Exception as e:
        print(f'{name}: {type(e).__name__} - {e}')
pd.DataFrame(rowsS)[['label','m','kappa','kappa_copy','alpha',
                     'alpha_over_kappa','kappa_copy_over_kappa']].round(3) if rowsS else 'skipped'

## 7. Verdict

In [ ]:
frames = []
for nm in ['A', 'X', 'R2', 'R']:
    df = globals().get(nm)
    if isinstance(df, pd.DataFrame) and len(df):
        frames.append(df)
for name, df in globals().get('T', {}).items():
    if len(df):
        d = df.copy(); d['label'] = name + ' ' + d['label']; frames.append(d)

ALL = pd.concat(frames, ignore_index=True, sort=False)
ALL.to_csv('alpha_kappa_scan.csv', index=False)

hits = ALL[(ALL.alpha_over_kappa < 0.2) & (ALL.copy_density > 0.01)]
print(f'{len(hits)} rows below 0.2 with a non-degenerate copy density:')
print(hits[['label','pattern','m','kappa','alpha','alpha_over_kappa','copy_density']]
      .round(3).to_string(index=False) if len(hits) else '  (none)')
print('wrote alpha_kappa_scan.csv')

**GO** if real rows — not only synthetic ones — appear in that list and the ratio trends downward
with size. Then the paper writes itself: the bracket lemma of section 1 explains the Array band, the
incidence ceiling of section 2 rules out the obvious domain class, cross-hyperedge triangles supply
the first natural separation family, and PredCount run on that domain supplies the speedup curve.
That is a DMKD paper: a data-mining primitive, a diagnostic a practitioner can compute, and a
measured map across real higher-order datasets.

**NO-GO** if only synthetic rows appear. Still publishable, but as a different paper — the two lemmas
plus the finding that the copy-free dense region is unreachable in practice. Weaker, and it would
need a second contribution to carry DMKD.